In [ ]:
!pip install catboost lightgbm xgboost tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/archive.zip"

Streaming output truncated to the last 5000 lines.
  inflating: training_setB/training_setB/p115003.psv  
  inflating: training_setB/training_setB/p115004.psv  
  inflating: training_setB/training_setB/p115005.psv  
  inflating: training_setB/training_setB/p115006.psv  
  inflating: training_setB/training_setB/p115007.psv  
  inflating: training_setB/training_setB/p115008.psv  
  inflating: training_setB/training_setB/p115009.psv  
  inflating: training_setB/training_setB/p115010.psv  
  inflating: training_setB/training_setB/p115011.psv  
  inflating: training_setB/training_setB/p115012.psv  
  inflating: training_setB/training_setB/p115013.psv  
  inflating: training_setB/training_setB/p115014.psv  
  inflating: training_setB/training_setB/p115015.psv  
  inflating: training_setB/training_setB/p115016.psv  
  inflating: training_setB/training_setB/p115017.psv  
  inflating: training_setB/training_setB/p115018.psv  
  inflating: training_setB/training_setB/p115019.psv  
  inflating: t

In [ ]:
import glob
import pandas as pd
from tqdm import tqdm

files = glob.glob("/content/training_setA/training/*.psv")
files += glob.glob("/content/training_setB/training_setB/*.psv")

print("Total patient files:", len(files))

df_list = []

for file in tqdm(files):
    patient_df = pd.read_csv(file, sep="|")
    patient_df["patient_id"] = file.split("/")[-1]
    df_list.append(patient_df)

df = pd.concat(df_list, ignore_index=True)

print("Dataset shape:", df.shape)
df.head()

Total patient files: 40336


100%|██████████| 40336/40336 [02:36<00:00, 258.42it/s]


Dataset shape: (1552210, 42)


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,88.0,100.0,35.90,124.0,84.0,72.0,12.0,NaN,-3.0,19.0,...,NaN,NaN,70.59,1,0.0,1.0,-306.53,2,0,p017471.psv
1,90.5,100.0,36.10,128.0,94.0,71.5,12.0,NaN,-1.0,NaN,...,NaN,NaN,70.59,1,0.0,1.0,-306.53,3,0,p017471.psv
2,92.0,100.0,36.45,114.0,88.0,70.0,13.0,NaN,NaN,NaN,...,NaN,NaN,70.59,1,0.0,1.0,-306.53,4,0,p017471.psv
3,97.0,99.5,36.80,127.0,97.0,76.0,23.0,NaN,NaN,NaN,...,NaN,NaN,70.59,1,0.0,1.0,-306.53,5,0,p017471.psv
4,96.0,98.5,37.40,114.0,89.0,71.0,22.0,NaN,-3.0,NaN,...,NaN,NaN,70.59,1,0.0,1.0,-306.53,6,0,p017471.psv


In [ ]:
df_model = df.drop(columns=["patient_id"])

df_model = df_model.fillna(df_model.median())

print("Processed dataset shape:", df_model.shape)

Processed dataset shape: (1552210, 41)


In [ ]:
X = df_model.drop(columns=["SepsisLabel"])
y = df_model["SepsisLabel"]

print(X.shape)
print(y.shape)

(1552210, 40)
(1552210,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1241768, 40)
Test shape: (310442, 40)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling done")

Scaling done


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    "MLP": MLPClassifier(max_iter=300),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "GaussianNB": GaussianNB()
}

In [ ]:
results = []

In [ ]:
for name, model in models.items():

    print("\n======================")
    print("Training:", name)
    print("======================")

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test)[:,1]
    else:
        prob = pred

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc = roc_auc_score(y_test, prob)

    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)
    print("ROC AUC  :", roc)

    results.append([name, acc, prec, rec, f1, roc])

    results_df = pd.DataFrame(
        results,
        columns=["Model","Accuracy","Precision","Recall","F1 Score","ROC AUC"]
    )

    results_df.to_csv("/content/drive/MyDrive/sepsis_results.csv", index=False)

    print("Saved results to Drive")


Training: MLP


NameError: name 'X_train' is not defined